# Partitioned training datasets

Incrementally grow a materialized Hopsworks training dataset without rewriting its existing data. Each batch is stored in Hive-style partitions derived from the feature view's event-time column, and time-range reads prune partitions that fall outside the requested window.

In this tutorial you will:

1. Create a feature group containing historical and newly arriving events.
2. Materialize an initial, unsplit Parquet training dataset.
3. Append a new batch and a late-arriving backfill.
4. Read sliding train and test windows from the same dataset version.
5. Refresh descriptive statistics after several appends.

> This API requires Hopsworks 5.1 or later.

In [ ]:
# Run this when the notebook environment does not already provide the SDK.
# %pip install -U "hopsworks[python]" --quiet

## Connect to Hopsworks

In [ ]:
import hopsworks
import pandas as pd

project = hopsworks.login()
fs = project.get_feature_store()

## Create event data

The feature group must define an event-time column for the training dataset to be addressable by time. This small example uses one row per customer per day.

In [ ]:
def make_events(start, end):
    dates = pd.date_range(start, end, freq="D", tz="UTC")
    rows = []
    for day_number, event_time in enumerate(dates):
        for customer_id in range(1, 6):
            rows.append(
                {
                    "customer_id": customer_id,
                    "event_time": event_time,
                    "transactions_7d": customer_id * 2 + day_number,
                    "balance": 1000.0 + customer_id * 100 - day_number * 5,
                    "churned": int((customer_id + day_number) % 7 == 0),
                }
            )
    return pd.DataFrame(rows)

events_fg = fs.get_or_create_feature_group(
    name="partitioned_training_events",
    version=1,
    description="Events for the partitioned training dataset tutorial",
    primary_key=["customer_id"],
    event_time="event_time",
)

events_fg.insert(make_events("2026-01-01", "2026-01-31"), wait=True)

## Create the initial partitioned training dataset

Create an **unsplit** training dataset. For time-series use cases, derive train and test windows when reading instead of appending to a time-series-split dataset, which is not supported.

`partition_precision` is fixed for the lifetime of this dataset version:

- `day` (default) gives precise daily windows.
- `month` or `year` creates fewer partitions for long histories, but reads select whole partitions at that precision.

In [ ]:
feature_view = fs.get_or_create_feature_view(
    name="partitioned_training_view",
    version=1,
    query=events_fg.select_all(),
    labels=["churned"],
)

td_version, creation_job = feature_view.create_training_data(
    description="Incrementally maintained customer-event training data",
    data_format="parquet",
    start_time="2026-01-01",
    end_time="2026-01-31 23:59:59",
    partition_precision="day",
)
print(f"Created training dataset version {td_version}")

The partition column is an internal storage detail and is not returned as a feature. Inspect the metadata to confirm the configured precision.

In [ ]:
training_dataset = feature_view.get_training_dataset(td_version)
print(training_dataset.name, training_dataset.version)
print("Partition precision:", training_dataset.partition_precision)

## Append a new increment

First write February events to the feature group. Then materialize only that event-time window with `insert_training_data`. The default `overwrite=False` preserves January and appends February to the same training dataset version.

In [ ]:
events_fg.insert(make_events("2026-02-01", "2026-02-28"), wait=True)

append_job = feature_view.insert_training_data(
    training_dataset_version=td_version,
    start_time="2026-02-01",
    end_time="2026-02-28 23:59:59",
)
print("February increment materialized")

Appending is not idempotent: materializing the same source window twice adds its rows twice. Track completed windows in the orchestration layer. Batches can otherwise arrive in any order, so backfills and late events are supported.

The next cell adds a late event for a date outside the windows already materialized, then backfills that day. If a backfill window overlaps an existing increment, its previously materialized source rows would be appended again.

In [ ]:
late_event = pd.DataFrame(
    [{
        "customer_id": 99,
        "event_time": pd.Timestamp("2025-12-15 12:00:00", tz="UTC"),
        "transactions_7d": 3,
        "balance": 750.0,
        "churned": 0,
    }]
)
events_fg.insert(late_event, wait=True)

backfill_job = feature_view.insert_training_data(
    training_dataset_version=td_version,
    start_time="2025-12-15",
    end_time="2025-12-15 23:59:59",
)
print("Late-event partition materialized")

## Read sliding train and test windows

Both bounds are inclusive and converted to UTC dates. Reads select whole partitions, so align bounds with `partition_precision`. With daily partitions, the following calls produce non-overlapping January and February windows.

In [ ]:
X_train, y_train = feature_view.get_training_data(
    training_dataset_version=td_version,
    start_time="2026-01-01",
    end_time="2026-01-31 23:59:59",
)

X_test, y_test = feature_view.get_training_data(
    training_dataset_version=td_version,
    start_time="2026-02-01",
    end_time="2026-02-28 23:59:59",
)

print("Train rows:", len(X_train), "Test rows:", len(X_test))
X_train.head()

Calling `get_training_data` without bounds returns all increments. A time-range read requires an event-time-partitioned materialized dataset; it is unavailable for in-memory datasets or counter-partitioned datasets created from a query without event time.

In [ ]:
X_all, y_all = feature_view.get_training_data(
    training_dataset_version=td_version
)
print("Rows across all increments:", len(X_all))

## Refresh descriptive statistics

Appends do not recompute statistics by default because doing so scans the full materialized dataset. Refresh them periodically, or use `compute_statistics=True` on an append when the append job is awaited. This refreshes descriptive statistics only; statistics used by model-dependent transformations remain pinned to those fitted when the dataset version was created.

In [ ]:
statistics = feature_view.compute_training_dataset_statistics(
    training_dataset_version=td_version
)
statistics

## Operational notes

- Incremental append supports materialized **Parquet** training datasets.
- Use an unsplit dataset for time-series data and derive time-based splits at read time. Appending to a time-series-split dataset is unsupported. Randomly split datasets can be appended, with each batch distributed across their splits.
- `overwrite=True` rewrites the entire version for the requested source window instead of adding an increment. Use it when transformation statistics must be refitted, or create a new dataset version.
- Without a left-feature-group event-time column, increments use counter partitions. They remain appendable but cannot be read by time range.
- Choose `month` or `year` precision only when fewer partitions matter more than fine-grained time-window reads.